Перед выполнением File/Add Input и добавить датасет: https://www.kaggle.com/datasets/mlmyavgav/msmarco


Конфиг токенизатора и получившиеся веса доступны на каггле:\
https://www.kaggle.com/code/whywhwh/splade/notebook?scriptVersionId=361087941

# Импорты

In [1]:
from pathlib import Path
import csv

import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, IterableDataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# Загрузка датасета

In [2]:
# Пути к подключённому Kaggle Dataset
DATA_DIR = Path("/kaggle/input/datasets/mlmyavgav/msmarco")

TRIPLES_PATH = DATA_DIR / "triples.train.small" / "triples.train.small.tsv"
QUERIES_DIR = DATA_DIR / "queries"
DEV_QUERIES_PATH = QUERIES_DIR / "queries.dev.tsv"
TRAIN_QUERIES_PATH = QUERIES_DIR / "queries.train.tsv"
EVAL_QUERIES_PATH = QUERIES_DIR / "queries.eval.tsv"
COLLECTION_PATH = DATA_DIR / "collection" / "collection.tsv"
QRELS_PATH = DATA_DIR / "qrels.dev.tsv"

# Проверьте, что все ожидаемые файлы доступны
required_files = [
    TRIPLES_PATH,
    DEV_QUERIES_PATH,
    TRAIN_QUERIES_PATH,
    EVAL_QUERIES_PATH,
    COLLECTION_PATH,
    QRELS_PATH,
]

missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Не найдены файлы:\n" + "\n".join(missing))

print("Все файлы найдены.")

# В этих TSV-файлах нет строки заголовков
def read_queries(path):
    return pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["qid", "query"],
        dtype={"qid": str, "query": str},
    )

# Dev небольшой, его можно прочитать целиком
dev_queries = read_queries(DEV_QUERIES_PATH)

dev_qrels = pd.read_csv(
    QRELS_PATH,
    sep="\t",
    header=None,
    names=["qid", "iteration", "pid", "relevance"],
    dtype={"qid": str, "iteration": str, "pid": str},
)

# Добавляем текст запроса к qrels для последующей оценки модели
dev_data = dev_qrels.merge(dev_queries, on="qid", how="left")

if dev_data["query"].isna().any():
    raise ValueError("Для некоторых qid из qrels не найден текст dev-запроса.")

print(f"Dev-запросов: {dev_queries['qid'].nunique():,}")
print(f"Dev qrels: {len(dev_qrels):,}")
display(dev_data.head())

# Train triplets читаем чанками, не загружая весь файл в RAM
TRIPLES_CHUNK_SIZE = 100_000

train_chunks = pd.read_csv(
    TRIPLES_PATH,
    sep="\t",
    header=None,
    names=["query", "positive", "negative"],
    chunksize=TRIPLES_CHUNK_SIZE,
    dtype=str,
)

first_train_chunk = next(train_chunks)
print(f"Размер одного train-чанка: {len(first_train_chunk):,}")
display(first_train_chunk.head())

# Пример обработки всех train-чанков:
# for chunk in pd.read_csv(
#     TRIPLES_PATH,
#     sep="\t",
#     header=None,
#     names=["query", "positive", "negative"],
#     chunksize=TRIPLES_CHUNK_SIZE,
#     dtype=str,
# ):
#     # Здесь обучайте модель на chunk
#     pass

# Коллекцию тоже читайте частями — целиком она большая
COLLECTION_CHUNK_SIZE = 100_000

collection_chunks = pd.read_csv(
    COLLECTION_PATH,
    sep="\t",
    header=None,
    names=["pid", "passage"],
    chunksize=COLLECTION_CHUNK_SIZE,
    dtype=str,
)

first_collection_chunk = next(collection_chunks)
print(f"Размер одного чанка коллекции: {len(first_collection_chunk):,}")
display(first_collection_chunk.head())

Все файлы найдены.
Dev-запросов: 101,093
Dev qrels: 59,273


,qid,iteration,pid,relevance,query
0,1102432,0,2026790,1,. what is a corporation?
1,1102431,0,7066866,1,why did rachel carson write an obligation to e...
2,1102431,0,7066867,1,why did rachel carson write an obligation to e...
3,1090282,0,7066900,1,symptoms of a dying mouse
4,39449,0,7066905,1,average number of lightning strikes per day


Размер одного train-чанка: 100,000


,query,positive,negative
0,is a little caffeine ok during pregnancy,We donât know a lot about the effects of caf...,It is generally safe for pregnant women to eat...
1,what fruit is native to australia,Passiflora herbertiana. A rare passion fruit n...,"The kola nut is the fruit of the kola tree, a ..."
2,how large is the canadian military,The Canadian Armed Forces. 1 The first large-...,The Canadian Physician Health Institute (CPHI)...
3,types of fruit trees,Cherry. Cherry trees are found throughout the ...,"The kola nut is the fruit of the kola tree, a ..."
4,how many calories a day are lost breastfeeding,"Not only is breastfeeding better for the baby,...","However, you still need some niacin each day; ..."


Размер одного чанка коллекции: 100,000


,pid,passage
0,0,The presence of communication amid scientific ...
1,1,The Manhattan Project and its atomic bomb help...
2,2,Essay on The Manhattan Project - The Manhattan...
3,3,The Manhattan Project was the name for a proje...
4,4,versions of each volume as well as complementa...


# Подготовка данных и батчинг

In [3]:
DATA = Path("/kaggle/input/datasets/mlmyavgav/msmarco")
TRIPLES = DATA / "triples.train.small/triples.train.small.tsv"
DEV_QUERIES = DATA / "queries/queries.dev.tsv"
QRELS = DATA / "qrels.dev.tsv"

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 4
MAX_STEPS = 1000
LR = 2e-5
LAMBDA_Q = 0.01
LAMBDA_D = 0.01

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [4]:
dev_queries = pd.read_csv(
    DEV_QUERIES, sep="\t", header=None, names=["qid", "query"]
)
dev_qrels = pd.read_csv(
    QRELS, sep="\t", header=None,
    names=["qid", "iteration", "pid", "relevance"]
)
print(dev_queries.shape, dev_qrels.shape)

(101093, 2) (59273, 4)


# Чтение датасета (построчно)

На kaggle для выполнения этой ячейки нужно активировать подключение к интернету для сессии (правая панель, session options, internet on). Для этого нужна вериф по номеру телефона и иногда просит по документам. Если не получается, в гугл колабе можно запустить.

In [5]:
class Triples(IterableDataset):  # Создаём потоковый датасет для чтения триплетов по одному.
    def __iter__(self):  # Этот метод вызывается, когда DataLoader запрашивает данные.
        with open(TRIPLES, encoding="utf-8") as f:  # Открываем файл с train-триплетами в UTF-8.
            for row in csv.reader(f, delimiter="\t"):  # Читаем файл построчно, разделяя поля табуляцией.
                if len(row) >= 3:  # Проверяем, что в строке есть query, positive и negative.
                    yield row[0], row[1], row[2]  # Возвращаем один триплет, не загружая весь файл в память.


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # Загружаем токенизатор выбранной BERT-модели.


def make_batch(rows):  # Собираем список триплетов в один batch для модели.
    queries, positives, negatives = zip(*rows)  # Разделяем триплеты на запросы, позитивные и негативные пассажи.

    def encode(texts):  # Преобразуем список текстов в токенизированный batch.
        return tokenizer(  # Вызываем токенизатор и возвращаем PyTorch-тензоры.
            list(texts),  # Передаём токенизатору список строк.
            padding=True,  # Дополняем короткие последовательности до длины самой длинной в batch.
            truncation=True,  # Обрезаем последовательности, превышающие MAX_LENGTH.
            max_length=MAX_LENGTH,  # Задаём максимальную длину последовательности.
            return_tensors="pt"  # Просим вернуть данные в виде PyTorch-тензоров.
        )

    return encode(queries), encode(positives), encode(negatives)  # Возвращаем три токенизированных batch.


train_loader = DataLoader(  # Создаём загрузчик, который будет собирать данные в batch-и.
    Triples(),  # Берём триплеты из потокового датасета выше.
    batch_size=BATCH_SIZE,  # Указываем число триплетов в одном batch.
    collate_fn=make_batch  # Передаём функцию для токенизации и сборки batch-а.
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# Функции

In [6]:
class SpladeMax(nn.Module):  # Описываем SPLADE-max как модель PyTorch.
    def __init__(self):  # Настраиваем и создаём слои модели.
        super().__init__()  # Инициализируем базовую часть nn.Module.
        # Загружаем BERT MLM checkpoint только для переноса его весов.
        # SPLADE forward ниже остаётся нашей ручной реализацией.
        pretrained = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)  # Загружаем предобученный BERT с MLM-головой.
        self.bert = pretrained.bert  # Берём из него BERT-энкодер.
        hidden = self.bert.config.hidden_size  # Получаем размер скрытого представления BERT.

        self.transform = nn.Linear(hidden, hidden)  # Создаём линейный слой для преобразования состояний токенов.
        self.layer_norm = nn.LayerNorm(  # Создаём LayerNorm для нормализации преобразованных состояний.
            hidden, eps=self.bert.config.layer_norm_eps  # Задаём размер вектора и epsilon из конфигурации BERT.
        )
        
        self.transform.load_state_dict(  # Копируем веса линейного слоя из предобученной MLM-головы.
            pretrained.cls.predictions.transform.dense.state_dict()  # Берём dense-слой MLM-головы BERT.
        )
        self.layer_norm.load_state_dict(  # Копируем параметры нормализации из предобученной MLM-головы.
            pretrained.cls.predictions.transform.LayerNorm.state_dict()  # Берём LayerNorm MLM-головы BERT.
        )
        self.vocab_bias = nn.Parameter(  # Создаём обучаемый bias для словарной проекции.
            pretrained.cls.predictions.bias.detach().clone()  # Копируем bias, отделив его от исходного графа вычислений.
        )
        del pretrained  # Удаляем временную ссылку на исходную MLM-модель.

    def forward(self, batch):  # Определяем прямой проход SPLADE для одного batch-а текстов.
        batch = {k: v.to(device) for k, v in batch.items()}  # Переносим входные тензоры на CPU или GPU.
        hidden = self.bert(**batch).last_hidden_state  # Получаем скрытые состояния всех токенов из BERT.
        hidden = self.layer_norm(F.gelu(self.transform(hidden)))  # Применяем Linear, GELU и LayerNorm к каждому токену.

        # Вес проекции разделяется с embedding-матрицей BERT.
        logits = F.linear(  # Проецируем состояния токенов в пространство всего словаря.
            hidden,  # Передаём преобразованные состояния токенов.
            self.bert.get_input_embeddings().weight,  # Используем embedding-веса BERT как веса словарной проекции.
            self.vocab_bias,  # Добавляем обучаемый bias каждого токена словаря.
        )
        x = torch.log1p(F.relu(logits))  # Применяем ReLU и log(1 + x), чтобы получить неотрицательные веса терминов.
        x = x.masked_fill(batch["attention_mask"].unsqueeze(-1) == 0, 0)  # Обнуляем веса, пришедшие из padding-токенов.
        return x.max(dim=1).values  # Max-pooling по позициям текста даёт один SPLADE-вектор на пример.


model = SpladeMax().to(device)  # Создаём SPLADE-модель и переносим её на выбранное устройство.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)  # Создаём AdamW для обновления параметров модели.

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
def flops_loss(x):  # Считаем FLOPS-регуляризацию для batch-а SPLADE-векторов.
    return x.mean(dim=0).square().sum()  # Усредняем активации по batch-у, возводим в квадрат и суммируем по словарю.


def loss_fn(q, pos, neg):  # Считаем общую функцию потерь для запросов и пассажей.
    scores = q @ pos.T  # Считаем оценки запросов для всех позитивных пассажей в batch-е.
    neg_scores = (q * neg).sum(dim=1, keepdim=True)  # Считаем оценку каждого запроса для его собственного негатива.
    scores = torch.cat([scores, neg_scores], dim=1)  # Добавляем оценки негативов к оценкам позитивных пассажей.
    labels = torch.arange(q.size(0), device=device)  # Указываем, что правильный позитив для каждого запроса стоит на диагонали.

    ranking = F.cross_entropy(scores, labels)  # Учим модель ставить соответствующий позитив выше остальных кандидатов.
    sparse = LAMBDA_Q * flops_loss(q) + LAMBDA_D * flops_loss(pos)  # Добавляем регуляризацию разреженности для запросов и позитивов.
    return ranking + sparse  # Возвращаем сумму ranking loss и FLOPS-регуляризации.

# Обучение

In [8]:
model.train()  # Переводим модель в режим обучения.

for step, (q_batch, pos_batch, neg_batch) in enumerate(train_loader, start=1):  # Перебираем batch-и триплетов, начиная нумерацию с 1.
    q = model(q_batch)  # Получаем SPLADE-векторы запросов.
    pos = model(pos_batch)  # Получаем SPLADE-векторы позитивных пассажей.
    neg = model(neg_batch)  # Получаем SPLADE-векторы негативных пассажей.

    loss = loss_fn(q, pos, neg)  # Считаем общую функцию потерь для текущего batch-а.
    optimizer.zero_grad()  # Обнуляем градиенты от предыдущего шага.
    loss.backward()  # Считаем градиенты параметров модели.
    optimizer.step()  # Обновляем параметры модели по градиентам.

    if step % 50 == 0:  # Каждые 50 шагов показываем текущий loss.
        print(step, round(loss.item(), 4))  # Печатаем номер шага и значение loss с четырьмя знаками после запятой.

    if step == MAX_STEPS:  # Проверяем, достигнуто ли заданное число шагов обучения.
        break  # Останавливаем цикл после MAX_STEPS.

50 6.8999
100 3.1809
150 1.6463
200 3.4877
250 1.9196
300 0.9468
350 1.3709
400 1.0287
450 1.2983
500 1.1604
550 0.4824
600 0.9888
650 0.7551
700 0.7683
750 0.2435
800 0.4683
850 0.2591
900 1.9663
950 0.7767
1000 0.8825


# Сохранение результатов

In [9]:
output = Path("/kaggle/working/splade_v2")
output.mkdir(exist_ok=True)
torch.save(model.state_dict(), output / "splade_max.pt")
tokenizer.save_pretrained(output / "tokenizer")
print("Сохранено в", output)


Сохранено в /kaggle/working/splade_v2
